# 03 - Tratamento dos dados de CNPJ - MG

Nesta etapa, o foco e entender a qualidade da base extraida para Minas Gerais e criar uma primeira versao limpa para analises posteriores.

A ideia aqui ainda nao e criar categorias de segmentos. Antes disso, este notebook identifica sujeiras, padroniza campos basicos, remove duplicidades obvias e salva uma base tratada em `data/processed/cnpj/cnpj_mg_limpo.csv`.

## 0. Objetivo da etapa

Principais entregas deste tratamento inicial:

- Carregar a base bruta gerada na extracao.
- Medir valores nulos, vazios, duplicados e codigos fora do escopo.
- Identificar textos com lixo comum, como espacos extras e marcadores sem informacao.
- Padronizar CNPJ, CEP, telefones, datas, capital social e textos principais.
- Salvar uma base limpa para as proximas etapas do projeto.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

## 1. Caminhos e configuracoes

A base de entrada vem da etapa 02. A saida deste notebook fica em `data/processed/cnpj`, separando a extracao bruta da base pronta para analise.

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "cnpj"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "cnpj"

INPUT_CSV = RAW_DIR / "cnpj_mg_extracao.csv"
OUTPUT_CSV = PROCESSED_DIR / "cnpj_mg_limpo.csv"
QUALITY_REPORT_CSV = PROCESSED_DIR / "cnpj_mg_relatorio_qualidade.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV, OUTPUT_CSV, QUALITY_REPORT_CSV

(WindowsPath('c:/Users/Dell/Desktop/Estudo Dados/b2b-prospect-radar/data/raw/cnpj/cnpj_mg_extracao.csv'),
 WindowsPath('c:/Users/Dell/Desktop/Estudo Dados/b2b-prospect-radar/data/processed/cnpj/cnpj_mg_limpo.csv'),
 WindowsPath('c:/Users/Dell/Desktop/Estudo Dados/b2b-prospect-radar/data/processed/cnpj/cnpj_mg_relatorio_qualidade.csv'))

## 2. Leitura da base bruta

Leio todas as colunas como texto para preservar zeros a esquerda em campos como CNPJ, CEP, DDD e codigos. Conversoes numericas e datas entram somente depois da limpeza inicial.

In [4]:
df_raw = pd.read_csv(
    INPUT_CSV,
    dtype=str,
    encoding="utf-8",
    low_memory=False,
)

print(f"Linhas: {len(df_raw):,}")
print(f"Colunas: {len(df_raw.columns):,}")
df_raw.head()

Linhas: 3,002,769
Colunas: 35


,cnpj_basico,cnpj_ordem,cnpj_dv,cnpj,nome_empresarial,nome_fantasia,matriz_filial,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,...,ddd_2,telefone_2,email,capital_social,porte_empresa,natureza_juridica,opcao_simples,opcao_mei,fonte_arquivo,data_extracao
0,66617294,0001,03,66617294000103,CONDOMINIO LUCCA RESIDENCE,NaN,1,02,20260120,00,...,37,35211680,CGONTIJO@CONTABILIDADEGONTIJO.COM.BR,"0,00",05,3085,NaN,NaN,Estabelecimentos0.zip,2026-05-13
1,09299839,0005,70,09299839000570,L S COMERCIO DE TINTAS LTDA,NaN,2,02,20260505,00,...,33,33312817,FINANCEIRO@SUACASADOPINTOR.COM.BR,"100000,00",05,2062,NaN,NaN,Estabelecimentos0.zip,2026-05-13
2,66617505,0001,08,66617505000108,66.617.505 RUBIA VITORIA DE OLIVEIRA,NaN,1,02,20260505,00,...,NaN,NaN,VRUBIA659@GMAIL.COM,"2000,00",01,2135,NaN,NaN,Estabelecimentos0.zip,2026-05-13
3,66617569,0001,09,66617569000109,66.617.569 NATALIANA DIAS ROSA,NaN,1,02,20260505,00,...,NaN,NaN,NDRSER.ADM@GMAIL.COM,"1,00",01,2135,NaN,NaN,Estabelecimentos0.zip,2026-05-13
4,66617581,0001,13,66617581000113,66.617.581 LUCAS ROSADO BARBOZA,NaN,1,02,20260505,00,...,NaN,NaN,E.LUCASRB19@GMAIL.COM,"3000,00",01,2135,NaN,NaN,Estabelecimentos0.zip,2026-05-13


## 3. Visao geral da qualidade

Antes de mexer nos dados, confiro tamanho da base, campos vazios e alguns exemplos. Essa leitura ajuda a separar problemas importantes de campos naturalmente incompletos, como complemento e segundo telefone.

In [5]:
overview = pd.DataFrame({
    "coluna": df_raw.columns,
    "tipo": [str(dtype) for dtype in df_raw.dtypes],
    "nulos": df_raw.isna().sum().values,
    "nulos_pct": (df_raw.isna().mean().values * 100).round(2),
    "valores_unicos": df_raw.nunique(dropna=True).values,
})

overview.sort_values("nulos_pct", ascending=False)

,coluna,tipo,nulos,nulos_pct,valores_unicos
32,opcao_mei,str,3002769,100.00,0
31,opcao_simples,str,3002769,100.00,0
26,telefone_2,str,2813442,93.69,100174
25,ddd_2,str,2811979,93.65,93
5,nome_fantasia,str,2203954,73.40,674544
17,complemento,str,1888865,62.90,153765
13,cnae_fiscal_secundaria,str,1257895,41.89,601540
27,email,str,253708,8.45,1892247
23,ddd_1,str,94979,3.16,206
24,telefone_1,str,94994,3.16,2039226


In [6]:
print("Duplicados por CNPJ:", df_raw.duplicated(subset=["cnpj"]).sum())
print("CNPJs unicos:", df_raw["cnpj"].nunique(dropna=True))

display(df_raw["uf"].value_counts(dropna=False).head(10))
display(df_raw["situacao_cadastral"].value_counts(dropna=False).head(10))
display(df_raw["matriz_filial"].value_counts(dropna=False).head(10))
display(df_raw["porte_empresa"].value_counts(dropna=False).head(10))

Duplicados por CNPJ: 0
CNPJs unicos: 3002769


uf
MG    3002769
Name: count, dtype: int64

situacao_cadastral
02    3002769
Name: count, dtype: int64

matriz_filial
1    2885935
2     116834
Name: count, dtype: int64

porte_empresa
01    2519994
05     351366
03     131409
Name: count, dtype: int64

## 4. Procura por lixo comum

Nesta parte, procuro valores que costumam atrapalhar analises: texto vazio disfaracado, espacos extras, marcadores como `NULL`, `N/A`, `SEM INFORMACAO` e campos numericos com caracteres inesperados.

In [7]:
EMPTY_TOKENS = {
    "",
    " ",
    "-",
    ".",
    "--",
    "NA",
    "N/A",
    "NULL",
    "NONE",
    "NAN",
    "SEM INFORMACAO",
    "SEM INFORMAÇÃO",
    "NAO INFORMADO",
    "NAO INFORMADA",
    "NÃO INFORMADO",
    "NÃO INFORMADA",
}

def count_empty_like(series: pd.Series) -> int:
    cleaned = series.astype("string").str.strip().str.upper()
    return int(cleaned.isin(EMPTY_TOKENS).sum())

empty_like_report = pd.DataFrame({
    "coluna": df_raw.columns,
    "vazios_disfarcados": [count_empty_like(df_raw[column]) for column in df_raw.columns],
})

empty_like_report[empty_like_report["vazios_disfarcados"] > 0].sort_values(
    "vazios_disfarcados",
    ascending=False,
)

,coluna,vazios_disfarcados
17,complemento,355
16,numero,118
5,nome_fantasia,94
18,bairro,59
27,email,35
15,logradouro,4


In [8]:
TEXT_COLUMNS = [
    "nome_empresarial",
    "nome_fantasia",
    "tipo_logradouro",
    "logradouro",
    "numero",
    "complemento",
    "bairro",
    "municipio_nome",
    "email",
]

whitespace_report = []
for column in TEXT_COLUMNS:
    if column not in df_raw.columns:
        continue
    values = df_raw[column].astype("string")
    whitespace_report.append({
        "coluna": column,
        "com_espaco_inicio_fim": int(values.str.match(r"^\s|.*\s$", na=False).sum()),
        "com_espaco_duplicado": int(values.str.contains(r"\s{2,}", regex=True, na=False).sum()),
    })

pd.DataFrame(whitespace_report).sort_values("com_espaco_duplicado", ascending=False)

,coluna,com_espaco_inicio_fim,com_espaco_duplicado
5,complemento,35326,363898
3,logradouro,3535,6688
6,bairro,498,1411
0,nome_empresarial,57,1405
4,numero,397,132
8,email,29,11
2,tipo_logradouro,0,2
1,nome_fantasia,1,0
7,municipio_nome,0,0


In [9]:
def only_digits(series: pd.Series) -> pd.Series:
    return series.astype("string").str.replace(r"\D+", "", regex=True)

numeric_quality = pd.DataFrame({
    "campo": ["cnpj", "cep", "ddd_1", "telefone_1", "ddd_2", "telefone_2"],
    "tamanho_esperado": [14, 8, 2, np.nan, 2, np.nan],
})

for column in numeric_quality["campo"]:
    digits = only_digits(df_raw[column])
    numeric_quality.loc[numeric_quality["campo"] == column, "vazios"] = int(digits.eq("").sum())
    numeric_quality.loc[numeric_quality["campo"] == column, "tamanhos_mais_comuns"] = str(digits.str.len().value_counts(dropna=False).head(5).to_dict())

numeric_quality

,campo,tamanho_esperado,vazios,tamanhos_mais_comuns
0,cnpj,14.0,0.0,{np.int64(14): 3002769}
1,cep,8.0,0.0,{np.int64(8): 3002769}
2,ddd_1,2.0,0.0,"{np.int64(2): 2893697, <NA>: 94979, np.int64(3..."
3,telefone_1,NaN,0.0,"{np.int64(8): 2889925, <NA>: 94994, np.int64(7..."
4,ddd_2,2.0,1.0,"{<NA>: 2811979, np.int64(2): 190775, np.int64(..."
5,telefone_2,NaN,0.0,"{<NA>: 2813442, np.int64(8): 189250, np.int64(..."


## 5. Funcoes de limpeza

As funcoes abaixo concentram a limpeza basica. Mantive a transformacao simples e rastreavel para facilitar ajustes depois que as primeiras analises mostrarem novos problemas.

In [10]:
def clean_text(series: pd.Series, uppercase: bool = True) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.str.replace(r"\s+", " ", regex=True)

    if uppercase:
        cleaned = cleaned.str.upper()

    cleaned = cleaned.mask(cleaned.str.upper().isin(EMPTY_TOKENS))
    return cleaned


def clean_digits(series: pd.Series, length: int | None = None) -> pd.Series:
    cleaned = only_digits(series)
    cleaned = cleaned.mask(cleaned.eq(""))
    if length is not None:
        cleaned = cleaned.str.zfill(length)
    return cleaned


def parse_receita_date(series: pd.Series) -> pd.Series:
    digits = clean_digits(series, length=8)
    return pd.to_datetime(digits, format="%Y%m%d", errors="coerce")


def parse_capital_social(series: pd.Series) -> pd.Series:
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(cleaned, errors="coerce")


def combine_phone(ddd: pd.Series, phone: pd.Series) -> pd.Series:
    ddd_clean = clean_digits(ddd, length=2)
    phone_clean = clean_digits(phone)
    combined = ddd_clean.fillna("") + phone_clean.fillna("")
    return combined.mask(combined.str.len() <= 2)

## 6. Aplicacao da limpeza

Aqui crio uma copia tratada da base. A limpeza remove sujeira de formato, mas nao cria classificacoes de segmento ainda, pois quero deixar preparado sem lixo nos dados.

In [11]:
df_clean = df_raw.copy()

for column in df_clean.select_dtypes(include="object").columns:
    df_clean[column] = clean_text(df_clean[column], uppercase=True)

for column, length in {
    "cnpj_basico": 8,
    "cnpj_ordem": 4,
    "cnpj_dv": 2,
    "cnpj": 14,
    "cep": 8,
    "municipio_codigo": 4,
    "cnae_fiscal_principal": 7,
    "ddd_1": 2,
    "ddd_2": 2,
}.items():
    if column in df_clean.columns:
        df_clean[column] = clean_digits(df_clean[column], length=length)

for column in ["telefone_1", "telefone_2", "natureza_juridica", "situacao_cadastral", "matriz_filial", "porte_empresa"]:
    if column in df_clean.columns:
        df_clean[column] = clean_digits(df_clean[column])

df_clean["telefone_1_completo"] = combine_phone(df_clean["ddd_1"], df_clean["telefone_1"])
df_clean["telefone_2_completo"] = combine_phone(df_clean["ddd_2"], df_clean["telefone_2"])

for column in ["data_situacao_cadastral", "data_inicio_atividade", "data_extracao"]:
    if column in df_clean.columns:
        if column == "data_extracao":
            df_clean[column] = pd.to_datetime(df_clean[column], errors="coerce")
        else:
            df_clean[column] = parse_receita_date(df_clean[column])

df_clean["capital_social"] = parse_capital_social(df_clean["capital_social"])

df_clean["email"] = df_clean["email"].str.lower()
df_clean.head()

C:\Users\Dell\AppData\Local\Temp\ipykernel_6124\2621660638.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df_clean.select_dtypes(include="object").columns:


,cnpj_basico,cnpj_ordem,cnpj_dv,cnpj,nome_empresarial,nome_fantasia,matriz_filial,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,...,email,capital_social,porte_empresa,natureza_juridica,opcao_simples,opcao_mei,fonte_arquivo,data_extracao,telefone_1_completo,telefone_2_completo
0,66617294,0001,03,66617294000103,CONDOMINIO LUCCA RESIDENCE,<NA>,1,02,2026-01-20,00,...,cgontijo@contabilidadegontijo.com.br,0.0,05,3085,<NA>,<NA>,ESTABELECIMENTOS0.ZIP,2026-05-13,3735211013,3735211680
1,09299839,0005,70,09299839000570,L S COMERCIO DE TINTAS LTDA,<NA>,2,02,2026-05-05,00,...,financeiro@suacasadopintor.com.br,100000.0,05,2062,<NA>,<NA>,ESTABELECIMENTOS0.ZIP,2026-05-13,3333313089,3333312817
2,66617505,0001,08,66617505000108,66.617.505 RUBIA VITORIA DE OLIVEIRA,<NA>,1,02,2026-05-05,00,...,vrubia659@gmail.com,2000.0,01,2135,<NA>,<NA>,ESTABELECIMENTOS0.ZIP,2026-05-13,3197288011,<NA>
3,66617569,0001,09,66617569000109,66.617.569 NATALIANA DIAS ROSA,<NA>,1,02,2026-05-05,00,...,ndrser.adm@gmail.com,1.0,01,2135,<NA>,<NA>,ESTABELECIMENTOS0.ZIP,2026-05-13,3199485180,<NA>
4,66617581,0001,13,66617581000113,66.617.581 LUCAS ROSADO BARBOZA,<NA>,1,02,2026-05-05,00,...,e.lucasrb19@gmail.com,3000.0,01,2135,<NA>,<NA>,ESTABELECIMENTOS0.ZIP,2026-05-13,3299431919,<NA>


## 7. Regras de filtro do escopo MG

Como a etapa anterior ja buscou empresas ativas de Minas Gerais, esta parte funciona como validacao e protecao. Se algum registro vier fora do escopo, ele e removido da base limpa.

In [12]:
before_scope_filter = len(df_clean)

df_clean = df_clean[
    (df_clean["uf"] == "MG")
    & (df_clean["situacao_cadastral"] == "02")
    & df_clean["cnpj"].notna()
    & (df_clean["cnpj"].str.len() == 14)
].copy()

removed_scope = before_scope_filter - len(df_clean)
print(f"Registros removidos por escopo/identificador invalido: {removed_scope:,}")
print(f"Registros restantes: {len(df_clean):,}")

Registros removidos por escopo/identificador invalido: 0
Registros restantes: 3,002,769


## 8. Duplicidades

A chave esperada da base e o CNPJ completo. Se houver repeticao, mantenho o primeiro registro para nao inflar contagens nas analises.

In [13]:
duplicated_cnpj = df_clean.duplicated(subset=["cnpj"], keep=False)
print(f"Linhas com CNPJ repetido: {duplicated_cnpj.sum():,}")

df_clean[df_clean.duplicated(subset=["cnpj"], keep=False)].sort_values("cnpj").head(20)

Linhas com CNPJ repetido: 0


,cnpj_basico,cnpj_ordem,cnpj_dv,cnpj,nome_empresarial,nome_fantasia,matriz_filial,situacao_cadastral,data_situacao_cadastral,motivo_situacao_cadastral,...,email,capital_social,porte_empresa,natureza_juridica,opcao_simples,opcao_mei,fonte_arquivo,data_extracao,telefone_1_completo,telefone_2_completo


In [14]:
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["cnpj"], keep="first").copy()
removed_duplicates = before_dedup - len(df_clean)

print(f"Duplicados removidos: {removed_duplicates:,}")
print(f"Total final: {len(df_clean):,}")

Duplicados removidos: 0
Total final: 3,002,769


## 9. Checagem depois da limpeza

Depois das transformacoes, gero uma nova leitura de qualidade para conferir se os principais problemas foram reduzidos e para documentar o estado da base limpa.

In [15]:
quality_after = pd.DataFrame({
    "coluna": df_clean.columns,
    "nulos": df_clean.isna().sum().values,
    "nulos_pct": (df_clean.isna().mean().values * 100).round(2),
    "valores_unicos": df_clean.nunique(dropna=True).values,
})

quality_after.sort_values("nulos_pct", ascending=False)

,coluna,nulos,nulos_pct,valores_unicos
32,opcao_mei,3002769,100.00,0
31,opcao_simples,3002769,100.00,0
36,telefone_2_completo,2813442,93.69,101350
26,telefone_2,2813442,93.69,100174
25,ddd_2,2811980,93.65,90
5,nome_fantasia,2204048,73.40,674500
17,complemento,1889220,62.92,142716
13,cnae_fiscal_secundaria,1257895,41.89,601540
27,email,253743,8.45,1874369
24,telefone_1,94994,3.16,2039217


In [16]:
print("Total de linhas:", f"{len(df_clean):,}")
print("CNPJs unicos:", f"{df_clean['cnpj'].nunique():,}")
print("UFs:")
display(df_clean["uf"].value_counts(dropna=False))

print("Situacao cadastral:")
display(df_clean["situacao_cadastral"].value_counts(dropna=False))

print("Top 10 municipios:")
display(df_clean["municipio_nome"].value_counts(dropna=False).head(10))

Total de linhas: 3,002,769
CNPJs unicos: 3,002,769
UFs:


uf
MG    3002769
Name: count, dtype: Int64

Situacao cadastral:


situacao_cadastral
02    3002769
Name: count, dtype: Int64

Top 10 municipios:


municipio_nome
BELO HORIZONTE          554390
UBERLANDIA              151109
CONTAGEM                112866
JUIZ DE FORA             92618
BETIM                    63449
MONTES CLAROS            60249
UBERABA                  52332
DIVINOPOLIS              47187
GOVERNADOR VALADARES     42843
IPATINGA                 38493
Name: count, dtype: Int64

## 10. Salvamento da base limpa

Salvo a base tratada e um relatorio simples de qualidade. As proximas etapas podem usar `cnpj_mg_limpo.csv` como entrada para categorizacao, SQL e dashboard.

In [17]:
df_clean.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
quality_after.to_csv(QUALITY_REPORT_CSV, index=False, encoding="utf-8")

print(f"Base limpa salva em: {OUTPUT_CSV}")
print(f"Relatorio de qualidade salvo em: {QUALITY_REPORT_CSV}")
print(f"Linhas salvas: {len(df_clean):,}")

Base limpa salva em: c:\Users\Dell\Desktop\Estudo Dados\b2b-prospect-radar\data\processed\cnpj\cnpj_mg_limpo.csv
Relatorio de qualidade salvo em: c:\Users\Dell\Desktop\Estudo Dados\b2b-prospect-radar\data\processed\cnpj\cnpj_mg_relatorio_qualidade.csv
Linhas salvas: 3,002,769


## 11. Proximos passos previstos

Com a limpeza inicial feita, a proxima etapa pode tratar a criacao de categorias de interesse comercial a partir de CNAE principal, CNAEs secundarios e termos nos nomes das empresas.